# Task 2: Normalize Breadboard Images

In [1]:
%load_ext autoreload
%autoreload 2

from breadboard_normalizer.normalizer import Normalizer, draw_corners, resize_width

In [2]:
normalizer = Normalizer(padding=[0.0, 0.0], output_resolution=[1024, 340])


In [3]:


normalizer.visualize_model("demo_images")

Failed to find corners at demo_images\20260428_212526.jpg
Failed to find corners at demo_images\20260428_212630.jpg


In [55]:
import cv2
import numpy as np
from PIL import Image

image = Image.open("example_training_images/4b6949fe-0ae765d4-69E2A987-8D4F-4075-A989-01B6BA302390.jpeg")


# normalizer assumes RGB
image = np.asarray(image)

# flip it to test the orientation detection
image = np.flipud(image)

norm, source_corners = normalizer.normalize_image(image)

if norm is None:
    print("Failed to normalize image")
else:
    # CV2 assumes BGR
    norm = np.flip(norm, axis=-1)
    image = np.flip(image, axis=-1)

    image = draw_corners(image, source_corners)

    norm = draw_corners(norm, normalizer.destination_corners)

    image = resize_width(image, normalizer.target_size[0])

    image = np.vstack([image, norm])

    cv2.imshow("image", image)
    cv2.waitKey(0)
    cv2.destroyAllWindows()

In [168]:
import cv2
cv2.destroyAllWindows()

In [183]:




def webcam_demo():
    # 1. Initialize the webcam (0 is typically the default camera)
    cap = cv2.VideoCapture(1)

    fourcc = cv2.VideoWriter_fourcc(*'mp4v')

    out = None

    if not cap.isOpened():
        print("Error: Could not open webcam.")
        return

    corner_history = np.zeros((4, 4, 2), dtype = np.float32)
    label_history = np.zeros((8), dtype=int)
    i = 0

    window_name = "Webcam Demo"
    cv2.namedWindow(window_name, cv2.WINDOW_NORMAL)

    while True:
        # 2. Capture frame-by-frame
        # ret is a boolean (True if frame is captured), frame is the image array
        ret, frame = cap.read()

        frame_scaled = resize_width(frame, normalizer.target_size[0])

        if not ret:
            print("Error: Can't receive frame. Exiting ...")
            break

        frame_rgb = np.flip(frame, axis=-1)
        # source_corners = normalizer.find_corners(frame_rgb)


        # norm_bgr = np.zeros((normalizer.target_size[1], normalizer.target_size[0], 3), dtype=np.uint8)


        # if source_corners is None:
        #     source_corners = corner_history[i % 4]
        # if source_corners is not None:
        #     i += 1

        # corner_history[i % 4] = source_corners

        # norm = normalizer.warp_image(frame_rgb, source_corners)

        # label = normalizer.breadboard_orientation_cv(norm)
        # # label = 'correct'

        # if label == 'flipped':
        #     label_history[i % 8] = 1
        # else:
        #     label_history[i % 8] = -1

        # if np.sum(label_history) > 0:
        #     norm = np.flipud(norm)
        #     source_corners = np.flip(source_corners, axis=0)

        # frame = draw_corners(frame, source_corners)
        # frame_scaled = resize_width(frame, normalizer.target_size[0])

        # norm_bgr = np.flip(norm, axis=-1)
        
        # cv2.imshow('Webcam Demo', np.vstack([frame_scaled, norm_bgr]))
        annotated = normalizer._show_annotated_image(frame_rgb, window_name)

        if out is None:
            out = cv2.VideoWriter('recordings/output.mp4', fourcc, 20.0, (annotated.shape[1], annotated.shape[0]))
        
        out.write(annotated)

        # 4. Press 'q' on the keyboard to exit
        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

    # 5. Release resources
    cap.release()

    out.release()
    cv2.destroyAllWindows()


In [184]:
webcam_demo()